In [ ]:
!pip install datasets==4.8.4 emoji==2.15.0 numpy==2.4.4 pandas==3.0.2 scikit-learn==1.8.0 torch==2.11.0 transformers==5.5.4 accelerate==1.13.0

In [ ]:
import pandas as pd
import numpy as np

# For machine learning tools and evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, balanced_accuracy_score


from sklearn.model_selection import train_test_split

import emoji
import re
import datasets
from transformers import TrainingArguments, Trainer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import warnings
from scipy.stats import chisquare

import torch

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("DTAI-KULeuven/robbert-2022-dutch-base")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("DTAI-KULeuven/robbert-2022-dutch-base")

In [ ]:
# Read data
data =  pd.read_csv("Final_labeled_data_reproducibility.csv")
# Classification only works when outcome variable is called labels
data[['labels']] = data[['label_mis']]
# Convert emoji to descriptions (in English) using emoji package
def no_emoji(text):
    text = emoji.demojize(text)
    return text
data['text'] = data['text'].apply(no_emoji)
#Remove urls, &gt, &lt and &amp, and [numbers] from text
#Removed [numbers] because there were meaningless numbers between brackets in the Tweets
def clean_text(text):
    text = re.sub(r'https?://\S+|www\.\S+|\r|\n|&gt.?| &lt.?|&amp.?|\[\d*\]', '', text)
    return text
data['text'] = data['text'].apply(clean_text)

#Display cleaned text
pd.set_option('display.max_rows', None)
pd.set_option('max_colwidth', None)

#One dataset with only text and labels to train BERT model
#One dataset with also year so I can later split dataset and evaluate metrics per year
#data_inf = data[['text', 'labels', 'id', 'year']]
data_inf = data[['text', 'labels', 'id']]
data = data[['text', 'labels']]


In [ ]:
max_length = 512
# set random seed for reproducibility
SEED_GLOBAL = 42
np.random.seed(SEED_GLOBAL)
#Set training directory
training_directory = "Roberta"

In [ ]:
# Train and test set for training model
df_train, df_test = train_test_split(data, random_state=42, test_size=0.25)

#Create identical test with text and labels plus year so performance metrics can be split by year
#df_train_inf, df_test_inf = train_test_split(data_inf, random_state=42, test_size=0.25)

In [ ]:
# convert pandas dataframes to Hugging Face dataset object to facilitate pre-processing
dataset = datasets.DatasetDict({
    "train": datasets.Dataset.from_pandas(df_train),
    "test": datasets.Dataset.from_pandas(df_test)
})

# tokenize
def tokenize(examples):
  return tokenizer(examples["text"], truncation=True, padding = 'max_length', max_length=512)  # max_length can be reduced to e.g. 256 to increase speed, but long texts will be cut off

dataset = dataset.map(tokenize, batched=True)

In [ ]:
# https://huggingface.co/docs/transformers/v5.5.4/en/main_classes/trainer#transformers.TrainingArguments
train_args = TrainingArguments(
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_ratio=0.06,
    weight_decay=0.1,
    seed=SEED_GLOBAL,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    eval_strategy="epoch",
    save_strategy = "epoch",
    report_to="all",
    output_dir=f'{training_directory}',
    logging_dir=f'{training_directory}',
)


In [ ]:
# Function to calculate metrics
# documentation on all metrics: https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics

def compute_metrics_standard(eval_pred):
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")

        labels = eval_pred.label_ids
        pred_logits = eval_pred.predictions
        preds_max = np.argmax(pred_logits, axis=1)

        # metrics
        precision_mis, recall_mis, f1_mis, _ = precision_recall_fscore_support(labels, preds_max, average= 'binary')
        precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(labels, preds_max, average='macro')
        precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(labels, preds_max, average='micro')
        acc_balanced = balanced_accuracy_score(labels, preds_max)
        acc_not_balanced = accuracy_score(labels, preds_max)

        metrics = {
            'accuracy': acc_not_balanced,
            'f1_macro': f1_macro,
            'accuracy_balanced': acc_balanced,
            'f1_micro': f1_micro,
            'precision_macro': precision_macro,
            'recall_macro': recall_macro,
            'precision_micro': precision_micro,
            'recall_micro': recall_micro,
            'precision_misinformation': precision_mis,
            'recall_misinformation': recall_mis,
            'f1_misinformation': f1_mis,
        }

        return metrics

In [ ]:
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics_standard
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model(output_dir = "Robbert2022Safe")

In [ ]:
model_path = "Robbert2022Safe"
model = AutoModelForSequenceClassification.from_pretrained(model_path)

In [ ]:
from transformers import pipeline

# documentation: https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.ZeroShotClassificationPipeline
pipe_classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    framework="pt"
)

In [ ]:
df_train_inf, df_test_inf = train_test_split(data_inf, random_state=42, test_size=0.25)

#Use df_test with date so data can be splitted on year
df_inference = df_test_inf[["text", "labels", "id"]].copy(deep=True)
text_lst = df_inference["text"].tolist()

#inference
pipe_output = pipe_classifier(
    text_lst,
    batch_size=64
)
print(pipe_output)

df_output = pd.DataFrame(pipe_output)

# add inference data to original dataframe
df_inference["label_text_pred"] = df_output["label"].tolist()
df_inference["label_text_pred_probability"] = df_output["score"].round(2).tolist()
#Print df_inference
df_inference

In [ ]:
df_inference['label_pred'] = df_inference['label_text_pred'].apply(lambda x: 1 if x == 'LABEL_1' else 0)


In [ ]:
df_inference.drop(columns = ['text']).to_csv("df_inference.csv")

In [48]:
!python --version

Python 3.12.13
